In [ ]:
import requests
import pandas as pd
import os
import time
import shutil

# ========================
# MONTAR GOOGLE DRIVE
# ========================
from google.colab import drive
drive.mount('/content/drive')

# Caminho base no Drive
BASE_DIR = "/content/drive/MyDrive/WebScraping/FEITO/SODIMAC/COLETA_SODIMAC/"
os.makedirs(BASE_DIR, exist_ok=True)

# ========================
# CONFIGURAÇÕES
# ========================
BASE_URL = "https://www.sodimac.com.br/s/search/v1/sobr/category/products"
CATEGORY_ID = "cat620002"  # pisos e revestimentos

OUTPUT_FILE_CSV = os.path.join(BASE_DIR, "sodimac_produtos.csv")
OUTPUT_FILE_XLSX = os.path.join(BASE_DIR, "sodimac_produtos.xlsx")
IMAGES_DIR = os.path.join(BASE_DIR, "sodimac_imagens")
ZIP_FILE = os.path.join(BASE_DIR, "sodimac_imagens.zip")

HEADERS = {"User-Agent": "Mozilla/5.0"}

# ========================
# FUNÇÃO PARA PEGAR PÁGINA
# ========================
def get_products(page=1):
    params = {
        "priceGroup": "1018",
        "zone": "35745",
        "currentpage": page,
        "channel": "kiosk",
        "region": "35",
        "city": "9575",
        "categoryId": CATEGORY_ID
    }
    try:
        r = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=10)
        r.raise_for_status()
        return r.json().get("data", {}).get("results", [])
    except Exception as e:
        print(f"❌ Erro na página {page}: {e}")
        return []

# ========================
# CARREGAR BACKUP (SE EXISTIR)
# ========================
if os.path.exists(OUTPUT_FILE_CSV):
    df_existente = pd.read_csv(OUTPUT_FILE_CSV, dtype=str)
    coletados = set(df_existente["Código"].astype(str))
    dados = df_existente.values.tolist()
    print(f"📂 Backup encontrado: {len(df_existente)} produtos já coletados.")
else:
    coletados = set()
    dados = []
    df_existente = pd.DataFrame()
    print("📂 Nenhum backup encontrado, começando do zero.")

# ========================
# COLETA PRODUTOS (SEM LIMITADOR)
# ========================
page = 1
novos = 0

while True:
    print(f"🔄 Coletando página {page}...")
    produtos = get_products(page)

    if not produtos:
        print("🚩 Nenhum produto retornado, fim da coleta.")
        break

    for p in produtos:
        codigo = str(p.get("productId", ""))

        # Pular se já coletado
        if codigo in coletados:
            continue

        nome = p.get("displayName", "")

        # Preço (EVENT > NORMAL > M2)
        preco = None
        for pr in p.get("prices", []):
            if pr.get("type") in ["EVENT", "NORMAL", "M2"]:
                preco = pr.get("price")
                break

        descricao = p.get("model", "")
        especificacoes = "; ".join(
            [f"{h['key']}: {h['value']}" for h in p.get("highlights", [])]
        )

        imagens = p.get("mediaUrls", [])
        imagem = imagens[-1] if imagens else None

        url = f"https://www.sodimac.com.br/sodimac-br/product/{codigo}/"
        loja = "Sodimac"

        dados.append([codigo, nome, preco, descricao, especificacoes, imagem, url, loja])
        coletados.add(codigo)
        novos += 1

    # Salvar progresso a cada página (CSV + Excel)
    colunas = ["Código","Nome","Preço Promocional","Descrição","Especificações","Imagem","URL","Loja"]
    df = pd.DataFrame(dados, columns=colunas)
    df.to_csv(OUTPUT_FILE_CSV, index=False, encoding="utf-8-sig")
    df.to_excel(OUTPUT_FILE_XLSX, index=False, engine="openpyxl")
    print(f"💾 Progresso salvo: {len(df)} produtos no CSV e Excel")

    page += 1
    time.sleep(1)

print(f"✅ Coleta finalizada! Total de {len(dados)} produtos salvos em {OUTPUT_FILE_CSV} e {OUTPUT_FILE_XLSX}")

# ========================
# BAIXAR IMAGENS
# ========================
os.makedirs(IMAGES_DIR, exist_ok=True)

for i, row in df.iterrows():
    codigo = row["Código"]
    img_url = row["Imagem"]

    if not img_url:
        print(f"⚠️ Produto {codigo} sem imagem.")
        continue

    img_path = os.path.join(IMAGES_DIR, f"{codigo}.jpg")

    # Pular se já existe
    if os.path.exists(img_path):
        continue

    try:
        r = requests.get(img_url, stream=True, timeout=10)
        if r.status_code == 200:
            with open(img_path, "wb") as f:
                shutil.copyfileobj(r.raw, f)
            print(f"📸 Imagem salva: {img_path}")
        else:
            print(f"❌ Erro ao baixar imagem {codigo}: {r.status_code}")
    except Exception as e:
        print(f"❌ Erro ao baixar imagem {codigo}: {e}")

# ========================
# COMPACTAR IMAGENS
# ========================
shutil.make_archive(IMAGES_DIR, "zip", IMAGES_DIR)
print(f"✅ Imagens salvas e compactadas em {ZIP_FILE}")


Mounted at /content/drive
📂 Backup encontrado: 10 produtos já coletados.
🔄 Coletando página 1...
💾 Progresso salvo: 28 produtos no CSV
🔄 Coletando página 2...
💾 Progresso salvo: 56 produtos no CSV
🔄 Coletando página 3...
💾 Progresso salvo: 84 produtos no CSV
🔄 Coletando página 4...
💾 Progresso salvo: 112 produtos no CSV
🔄 Coletando página 5...
💾 Progresso salvo: 133 produtos no CSV
🔄 Coletando página 6...
💾 Progresso salvo: 155 produtos no CSV
🔄 Coletando página 7...
💾 Progresso salvo: 183 produtos no CSV
🔄 Coletando página 8...
💾 Progresso salvo: 211 produtos no CSV
🔄 Coletando página 9...
💾 Progresso salvo: 232 produtos no CSV
🔄 Coletando página 10...
💾 Progresso salvo: 252 produtos no CSV
🔄 Coletando página 11...
💾 Progresso salvo: 280 produtos no CSV
🔄 Coletando página 12...
💾 Progresso salvo: 308 produtos no CSV
🔄 Coletando página 13...
💾 Progresso salvo: 336 produtos no CSV
🔄 Coletando página 14...
💾 Progresso salvo: 364 produtos no CSV
🔄 Coletando página 15...
💾 Progresso salvo: